# Agentic AI as a System for Engineering Research
### INTCEC 2026 Hands-On Workshop — Chicago, September 2026

**What you will build in 40 minutes:** a three-stage research assistant made of open-source agents that
1. **Part A** — searches and screens literature (arXiv),
2. **Part B** — explores an engineering dataset and fits a model,
3. **Part C** — drafts a structured mini-report from the outputs of A and B.

**Stack:** [smolagents](https://github.com/huggingface/smolagents) (Hugging Face, open source) + a free LLM API key (Gemini or Groq). Everything runs in this Colab notebook — nothing to install on your laptop.

> ⚠️ Before the workshop: get a free API key (no credit card) from **[Google AI Studio](https://aistudio.google.com/apikey)** (recommended) or **[Groq](https://console.groq.com/keys)**. See the pre-workshop email.


## 0. Setup (≈3 min)

Run the two cells below. The first installs packages (~1 min), the second asks for your API key (it is kept only in this session's memory, not saved).

> **Note:** free API tiers are rate-limited (Gemini Flash ≈10 requests/min). Agents make several calls per run, so an occasional pause or `429` is normal — wait a minute and re-run the cell.

In [ ]:
%pip -q install "smolagents[litellm]" pandas matplotlib scikit-learn openpyxl xlrd
print("Install done ✅")

In [ ]:
import getpass, os

PROVIDER = "gemini"   # change to "groq" if you brought a Groq key

def read_key(prompt):
    key = getpass.getpass(prompt).strip()
    if not key or not key.isascii() or any(ch.isspace() for ch in key) or len(key) > 100:
        raise ValueError(
            "That doesn't look like an API key (extra text may have been pasted). "
            "Use the COPY BUTTON next to the key on the provider's page, then re-run this cell.")
    return key

if PROVIDER == "gemini":
    os.environ["GEMINI_API_KEY"] = read_key("Paste your Gemini API key: ")
    MODEL_ID = "gemini/gemini-3.5-flash"   # if this 404s, run the model-list cell below
else:
    os.environ["GROQ_API_KEY"] = read_key("Paste your Groq API key: ")
    MODEL_ID = "groq/llama-3.3-70b-versatile"

from smolagents import LiteLLMModel
model = LiteLLMModel(model_id=MODEL_ID)
print(f"Model ready: {MODEL_ID} \u2705")

In [ ]:
# OPTIONAL — run only if the model check below fails with a 404 ("model not available").
# Lists the Gemini models YOUR key can use; pick one and update MODEL_ID.
import os, requests
if PROVIDER == "gemini":
    r = requests.get("https://generativelanguage.googleapis.com/v1beta/models",
                     headers={"x-goog-api-key": os.environ["GEMINI_API_KEY"]})
    names = [m["name"].removeprefix("models/") for m in r.json().get("models", [])
             if "generateContent" in m.get("supportedGenerationMethods", [])]
    print("\n".join(names))
    # then e.g.:  MODEL_ID = "gemini/" + names[0]; model = LiteLLMModel(model_id=MODEL_ID)

### Quick sanity check
One plain LLM call — **no agent yet**. Note what it can and cannot do: it answers from parametric memory, cannot fetch papers, run code, or verify anything. That gap is what agents close.

In [ ]:
resp = model([{"role": "user", "content": "In two sentences: what is an AI agent, vs. a plain chatbot?"}])
print(resp.content)

---
## Part A — Literature review agent (≈10 min)

**The agent loop:** an LLM that repeatedly *thinks → acts (calls a tool / runs code) → observes the result* until it decides the task is done. Below we give it one tool: a real literature search — arXiv first, with automatic fallback to OpenAlex (both public APIs, no key needed). Because the citations come from a live API call, **every returned ID is verifiable** — a key defense against hallucinated references.

In [ ]:
import json, time
import urllib.request, urllib.parse, urllib.error
import xml.etree.ElementTree as ET
from smolagents import tool

# arXiv throttles/slows for shared Colab IPs, so: identify ourselves, retry,
# and fall back to OpenAlex (free, no key, built for programmatic use).
UA = {"User-Agent": "INTCEC2026-agents-workshop/1.0 (contact: 1p1r1m1@gmail.com)"}

def _fetch(url, timeout=45):
    req = urllib.request.Request(url, headers=UA)
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

def _arxiv(query, max_results):
    url = ("https://export.arxiv.org/api/query?search_query=all:"
           + urllib.parse.quote(query)
           + f"&start=0&max_results={max_results}&sortBy=submittedDate&sortOrder=descending")
    root = ET.fromstring(_fetch(url))
    ns = {"a": "http://www.w3.org/2005/Atom"}
    out = []
    for e in root.findall("a:entry", ns):
        title = " ".join(e.find("a:title", ns).text.split())
        summary = " ".join(e.find("a:summary", ns).text.split())[:400]
        authors = ", ".join(a.find("a:name", ns).text for a in e.findall("a:author", ns)[:3])
        arxiv_id = e.find("a:id", ns).text.split("/abs/")[-1]
        published = e.find("a:published", ns).text[:10]
        out.append(f"[arXiv:{arxiv_id}] ({published}) {title} — {authors}. Abstract: {summary}")
    return out

def _openalex(query, max_results):
    url = ("https://api.openalex.org/works?search=" + urllib.parse.quote(query)
           + f"&per-page={max_results}&mailto=1p1r1m1@gmail.com")
    data = json.loads(_fetch(url))
    out = []
    for w in data.get("results", []):
        title = w.get("title") or "(untitled)"
        year = w.get("publication_year", "?")
        authors = ", ".join(a["author"]["display_name"] for a in w.get("authorships", [])[:3])
        doi = (w.get("doi") or w.get("id") or "").replace("https://doi.org/", "")
        inv = w.get("abstract_inverted_index")
        abstract = ""
        if inv:
            pos = sorted((p, word) for word, ps in inv.items() for p in ps)
            abstract = " ".join(word for _, word in pos)[:400]
        out.append(f"[DOI:{doi}] ({year}) {title} — {authors}. Abstract: {abstract}")
    return out

@tool
def search_papers(query: str, max_results: int = 8) -> str:
    """Search the scholarly literature (arXiv first, OpenAlex as fallback) and return matching papers.

    Args:
        query: Search terms, e.g. 'graph neural network structural health monitoring'.
        max_results: Number of papers to return (1-20).
    """
    for attempt in range(2):  # arXiv: 2 tries
        try:
            res = _arxiv(query, max_results)
            if res:
                return "\n\n".join(res)
        except Exception:
            time.sleep(3)
    try:  # fallback: OpenAlex
        res = _openalex(query, max_results)
        if res:
            return "SOURCE: OpenAlex (arXiv unavailable)\n\n" + "\n\n".join(res)
    except Exception as e:
        return f"Both arXiv and OpenAlex failed ({e}). Wait ~30 s and try again."
    return "No results found. Try broader search terms."

# quick test of the tool itself (no LLM involved):
print(search_papers("agentic AI engineering design", max_results=2)[:600])

In [ ]:
from smolagents import ToolCallingAgent

lit_agent = ToolCallingAgent(tools=[search_papers], model=model, max_steps=6)

RESEARCH_TOPIC = "large language model agents for engineering design optimization"  # ← change to YOUR topic

lit_review = lit_agent.run(f"""
You are a research assistant. Do a mini literature review on: "{RESEARCH_TOPIC}".
1. Search the literature (try 2 different query phrasings).
2. Select the 5 most relevant papers.
3. Return a markdown table: ID (arXiv or DOI) | Year | Title | One-line contribution | Method type.
4. Below the table, write a 5-sentence synthesis: what is the trend, and what gap remains?
Only cite papers returned by the tool — never invent references.
""")
from IPython.display import Markdown, display
display(Markdown(str(lit_review)))

**🔧 Your turn (5 min):** change `RESEARCH_TOPIC` to *your own research area* and re-run. Then try adding a constraint to the prompt, e.g. "only papers using real experimental data" — watch how the agent re-plans its searches.

**Verify:** pick one ID from the table and open `arxiv.org/abs/<id>` (or `doi.org/<doi>`). It exists — because it came from the API, not the model's memory.

---
## Part B — Data analysis agent (≈12 min)

Now a **CodeAgent**: instead of calling fixed tools, it *writes and executes Python* in a sandbox — pandas, sklearn, matplotlib. We use the UCI **Concrete Compressive Strength** dataset (1,030 mixes, 8 ingredients/age features — a classic civil-engineering regression problem).

In [ ]:
import pandas as pd, numpy as np

URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls"
try:
    df = pd.read_excel(URL)
    df.columns = ["cement","slag","fly_ash","water","superplasticizer",
                  "coarse_agg","fine_agg","age_days","strength_MPa"]
    print("Loaded UCI concrete dataset ✅", df.shape)
except Exception as ex:
    print("Download failed, generating synthetic fallback:", ex)
    rng = np.random.default_rng(0)
    n = 1030
    df = pd.DataFrame({
        "cement": rng.uniform(100, 550, n), "slag": rng.uniform(0, 360, n),
        "fly_ash": rng.uniform(0, 200, n), "water": rng.uniform(120, 250, n),
        "superplasticizer": rng.uniform(0, 32, n), "coarse_agg": rng.uniform(800, 1150, n),
        "fine_agg": rng.uniform(590, 995, n), "age_days": rng.choice([3,7,14,28,56,90,180,365], n),
    })
    df["strength_MPa"] = (0.11*df.cement + 0.08*df.slag + 0.06*df.fly_ash - 0.20*df.water
                          + 0.6*df.superplasticizer + 8*np.log(df.age_days) + rng.normal(0, 5, n)).clip(2, 85)
df.head()

In [ ]:
from smolagents import CodeAgent

data_agent = CodeAgent(
    tools=[],
    model=model,
    additional_authorized_imports=["pandas", "numpy", "matplotlib", "matplotlib.pyplot", "sklearn", "sklearn.*"],
    max_steps=10,
)

analysis = data_agent.run("""
You are given a pandas DataFrame `df` of concrete mixes (features) and compressive strength `strength_MPa` (target).
1. Briefly profile the data (shape, ranges, missing values).
2. Which 3 features correlate most with strength?
3. Fit a RandomForestRegressor (train/test split, random_state=42) and report test R² and RMSE.
4. Make ONE matplotlib figure: predicted vs. actual strength with a 45° reference line, labeled axes with units.
5. End with a 3-sentence engineering interpretation.
""", additional_args={"df": df})
print(analysis)

**🔧 Your turn (5 min):** ask a *different* engineering question of the same agent, e.g.:
- "What water/cement ratio maximizes 28-day strength in this data? Support with a plot."
- "Is fly ash an effective partial cement replacement here? Quantify."

**Discussion point:** the agent chose the split, metric, and model settings. For a paper you must *audit* that — read the code it wrote (printed in the step logs above). Agents accelerate analysis; they do not replace methodological judgment.

---
## Part C — Drafting agent + the "system" view (≈10 min)

The point of the workshop title: agents become powerful as a **system** — outputs of one feed the next. We pass Part A's review and Part B's results to a writing agent.

In [ ]:
writer_agent = ToolCallingAgent(tools=[], model=model, max_steps=3)

draft = writer_agent.run(f"""
You are drafting a section of a short engineering research paper (IEEE conference style, formal, no hype).
Using ONLY the material below, write:
1. A 'Related Work' paragraph (cite papers as [arXiv:ID]).
2. A 'Preliminary Results' paragraph reporting the model performance with numbers.
3. Two sentences of 'Future Work'.

=== LITERATURE REVIEW (from Part A) ===
{lit_review}

=== ANALYSIS RESULTS (from Part B) ===
{analysis}
""")
display(Markdown(str(draft)))

**🔧 Your turn:** re-run with a different style instruction ("for a grant proposal's preliminary-data section", "as a plain-language summary for a project sponsor"). Same evidence, different rhetorical target.

### Stretch goal — a manager agent (if time permits)
smolagents lets one agent delegate to others (`managed_agents=[...]`) — a true multi-agent system. Sketch:

In [ ]:
# Stretch (runs ~2-4 min; skip if short on time)
from smolagents import ToolCallingAgent, CodeAgent

lit_worker = ToolCallingAgent(tools=[search_papers], model=model, max_steps=5,
                              name="literature_agent",
                              description="Searches the scholarly literature and summarizes papers on a given topic.")
manager = CodeAgent(tools=[], model=model, managed_agents=[lit_worker], max_steps=6)
out = manager.run("Ask literature_agent for 3 recent papers on 'digital twins for bridge monitoring', "
                  "then rank them by likely relevance to a state DOT and justify the ranking.")
print(out)

---
## Wrap-up: using this responsibly in research

- **Verify citations.** Only trust references that came from a live database call (arXiv, Semantic Scholar, PubMed APIs). LLM-generated citations without tool grounding are unreliable.
- **Audit generated code.** Step logs show every line the agent ran — read them before results go in a paper.
- **Reproducibility.** Pin package versions, set seeds, save agent logs; agent runs are stochastic.
- **Disclosure.** IEEE currently requires disclosure of AI-generated text in submissions and prohibits listing AI as an author — check your venue's current policy before submitting.
- **Data privacy.** Free API tiers may use your prompts for training. Do not paste unpublished data/ideas you must protect; use local models (Ollama) for sensitive work.

### Where to go next
- smolagents docs: https://huggingface.co/docs/smolagents · LangGraph: https://langchain-ai.github.io/langgraph/ · CrewAI: https://docs.crewai.com · AutoGen: https://microsoft.github.io/autogen/
- Local/private: Ollama (https://ollama.com) + any of the above.
- Commercial agentic tools with the same loop built-in: Claude Code, OpenAI Codex, Gemini CLI, GitHub Copilot agents.

*Workshop materials: INTCEC 2026 — Agentic AI as a System for Engineering Research.*